# Fast tokenizers' special powers (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example = "My name is Bright and I work at Hugging Face in Brooklyn."
encoding = tokenizer(example)
print(type(encoding))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [3]:
tokenizer.is_fast

True

In [4]:
encoding.is_fast

True

In [9]:
encoding.tokens()

['[CLS]',
 'My',
 'name',
 'is',
 'Bright',
 'and',
 'I',
 'work',
 'at',
 'Hu',
 '##gging',
 'Face',
 'in',
 'Brooklyn',
 '.',
 '[SEP]']

In [10]:
encoding.word_ids()

[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 8, 9, 10, 11, 12, None]

In [11]:
start, end = encoding.word_to_chars(3)
example[start:end]

'Bright'

In [12]:
from transformers import pipeline

token_classifier = pipeline("token-classification")
token_classifier("My name is Thanapol and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

[{'entity': 'I-PER',
  'score': np.float32(0.9728421),
  'index': 4,
  'word': 'Than',
  'start': 11,
  'end': 15},
 {'entity': 'I-PER',
  'score': np.float32(0.95295376),
  'index': 5,
  'word': '##ap',
  'start': 15,
  'end': 17},
 {'entity': 'I-PER',
  'score': np.float32(0.8763486),
  'index': 6,
  'word': '##ol',
  'start': 17,
  'end': 19},
 {'entity': 'I-ORG',
  'score': np.float32(0.96838266),
  'index': 11,
  'word': 'Hu',
  'start': 34,
  'end': 36},
 {'entity': 'I-ORG',
  'score': np.float32(0.97736174),
  'index': 12,
  'word': '##gging',
  'start': 36,
  'end': 41},
 {'entity': 'I-ORG',
  'score': np.float32(0.9868422),
  'index': 13,
  'word': 'Face',
  'start': 42,
  'end': 46},
 {'entity': 'I-LOC',
  'score': np.float32(0.9926369),
  'index': 15,
  'word': 'Brooklyn',
  'start': 50,
  'end': 58}]

In [14]:
from transformers import pipeline

token_classifier = pipeline("token-classification", aggregation_strategy="simple")
token_classifier("My name is Thanapol and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9340482),
  'word': 'Thanapol',
  'start': 11,
  'end': 19},
 {'entity_group': 'ORG',
  'score': np.float32(0.97752887),
  'word': 'Hugging Face',
  'start': 34,
  'end': 46},
 {'entity_group': 'LOC',
  'score': np.float32(0.9926369),
  'word': 'Brooklyn',
  'start': 50,
  'end': 58}]

In [15]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

example = "My name is Thanapol and I work at Hugging Face in Brooklyn."
inputs = tokenizer(example, return_tensors="pt")
outputs = model(**inputs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
print(inputs["input_ids"].shape)
print(outputs.logits.shape)

torch.Size([1, 18])
torch.Size([1, 18, 9])


In [17]:
import torch

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
predictions = outputs.logits.argmax(dim=-1)[0].tolist()
print(predictions)

[0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 6, 6, 6, 0, 8, 0, 0]


In [18]:
model.config.id2label

{0: 'O',
 1: 'B-MISC',
 2: 'I-MISC',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

In [19]:
results = []
tokens = inputs.tokens()

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        results.append(
            {"entity": label, "score": probabilities[idx][pred], "word": tokens[idx]}
        )

print(results)

[{'entity': 'I-PER', 'score': 0.9728420972824097, 'word': 'Than'}, {'entity': 'I-PER', 'score': 0.9529537558555603, 'word': '##ap'}, {'entity': 'I-PER', 'score': 0.8763487935066223, 'word': '##ol'}, {'entity': 'I-ORG', 'score': 0.9683825373649597, 'word': 'Hu'}, {'entity': 'I-ORG', 'score': 0.9773618578910828, 'word': '##gging'}, {'entity': 'I-ORG', 'score': 0.9868422150611877, 'word': 'Face'}, {'entity': 'I-LOC', 'score': 0.9926369190216064, 'word': 'Brooklyn'}]


In [20]:
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
inputs_with_offsets["offset_mapping"]

[(0, 0),
 (0, 2),
 (3, 7),
 (8, 10),
 (11, 15),
 (15, 17),
 (17, 19),
 (20, 23),
 (24, 25),
 (26, 30),
 (31, 33),
 (34, 36),
 (36, 41),
 (42, 46),
 (47, 49),
 (50, 58),
 (58, 59),
 (0, 0)]

In [21]:
example[12:14]

'ha'

In [22]:
results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        start, end = offsets[idx]
        results.append(
            {
                "entity": label,
                "score": probabilities[idx][pred],
                "word": tokens[idx],
                "start": start,
                "end": end,
            }
        )

print(results)

[{'entity': 'I-PER', 'score': 0.9728420972824097, 'word': 'Than', 'start': 11, 'end': 15}, {'entity': 'I-PER', 'score': 0.9529537558555603, 'word': '##ap', 'start': 15, 'end': 17}, {'entity': 'I-PER', 'score': 0.8763487935066223, 'word': '##ol', 'start': 17, 'end': 19}, {'entity': 'I-ORG', 'score': 0.9683825373649597, 'word': 'Hu', 'start': 34, 'end': 36}, {'entity': 'I-ORG', 'score': 0.9773618578910828, 'word': '##gging', 'start': 36, 'end': 41}, {'entity': 'I-ORG', 'score': 0.9868422150611877, 'word': 'Face', 'start': 42, 'end': 46}, {'entity': 'I-LOC', 'score': 0.9926369190216064, 'word': 'Brooklyn', 'start': 50, 'end': 58}]


In [25]:
example[11:19]

'Thanapol'

In [24]:
import numpy as np

results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

idx = 0
while idx < len(predictions):
    pred = predictions[idx]
    label = model.config.id2label[pred]
    if label != "O":
        # Remove the B- or I-
        label = label[2:]
        start, _ = offsets[idx]

        # Grab all the tokens labeled with I-label
        all_scores = []
        while (
            idx < len(predictions)
            and model.config.id2label[predictions[idx]] == f"I-{label}"
        ):
            all_scores.append(probabilities[idx][pred])
            _, end = offsets[idx]
            idx += 1

        # The score is the mean of all the scores of the tokens in that grouped entity
        score = np.mean(all_scores).item()
        word = example[start:end]
        results.append(
            {
                "entity_group": label,
                "score": score,
                "word": word,
                "start": start,
                "end": end,
            }
        )
    idx += 1

print(results)

[{'entity_group': 'PER', 'score': 0.9340482155481974, 'word': 'Thanapol', 'start': 11, 'end': 19}, {'entity_group': 'ORG', 'score': 0.9775288701057434, 'word': 'Hugging Face', 'start': 34, 'end': 46}, {'entity_group': 'LOC', 'score': 0.9926369190216064, 'word': 'Brooklyn', 'start': 50, 'end': 58}]
